# TrainLM on TPU

This is the same Hugging Face-like workflow an end user runs. The first cells clone and refresh the requested TrainLM branch, then install the package and dependencies. Choose a TPU runtime and local cache/output directories below, then call `trainer.train()`.

The throughput run uses a 135M Llama-shaped geometry: 8 layers, hidden size 1024, sequence length 2048, batch 8 per replica, and gradient accumulation 8. Chunked output projection keeps the larger microbatch memory-bounded while halving host-driven microsteps per update. TrainLM handles TPU discovery, world size, worker launch, ranks, preflight, caching, distributed data ownership, and structured results under the hood.


In [ ]:
print("[Notebook 1/8] Cloning the TrainLM branch...")
!git clone --branch milestone/m10-m12-kernels-parity --single-branch https://github.com/Dhiraj309/TrainLM.git


In [ ]:
%cd TrainLM
print("[Notebook 2/8] Repository selected.")


In [ ]:
%%bash
set -euo pipefail
echo "[Notebook 3/8] Refreshing the requested branch from origin..."
cd /kaggle/working/TrainLM

BRANCH="milestone/m10-m12-kernels-parity"

git fetch origin "$BRANCH"
git checkout -B "$BRANCH" "origin/$BRANCH"

echo "Current TrainLM revision:"
git log -1 --oneline
git rev-parse HEAD
echo "[Notebook 3/8] Checkout is current."


## 1. Install and restart once

Kaggle preinstalls TensorFlow and mismatched vision/audio wheels that can initialize or conflict with Torch/XLA in the notebook process. TrainLM text pretraining does not use them. Run the cleanup/install cell, wait for it to finish, then use **Restart Session** before running section 2. Do not continue in the same kernel.


In [ ]:
print("[Notebook 4/8] Installing the pinned TPU environment...")
%pip uninstall -y tensorflow tensorflow-cpu tensorflow-gpu tensorflow-intel tensorflow-rocm tf-keras keras torchvision torchaudio
%pip install -e ".[tpu-xla]" -c constraints/tpu-xla-2.9.txt

# Some Kaggle images leave an orphaned tensorflow/ package directory after
# its distribution metadata is removed. Remove only that exact package
# directory; the required restart below clears any already-loaded module.
import importlib.util
from pathlib import Path
import shutil
_tensorflow_spec = importlib.util.find_spec("tensorflow")
if _tensorflow_spec is not None and _tensorflow_spec.submodule_search_locations:
    for _location in _tensorflow_spec.submodule_search_locations:
        _path = Path(_location)
        if _path.name == "tensorflow" and "site-packages" in _path.parts:
            print("Removing orphan TensorFlow package directory:", _path)
            shutil.rmtree(_path, ignore_errors=False)
print("[Notebook 4/8] Install complete. Restart Session before continuing.")


## 2. Your inputs

Choose how many numbered shards to use. `(0, TRAIN_SHARD_STOP)` is end-exclusive, so `1` downloads shard `00000`. TrainLM resolves `main` to its immutable Hub commit and saves the file in `DATA_CACHE_DIR`.

The reference model is initialized from a Hugging Face Llama configuration whose 32,064-token vocabulary matches the LaughLM packed-token metadata. This avoids the deeper 30-layer, 49,152-vocabulary SmolLM graph that was slower to compile and did not share the dataset tokenizer mapping.


In [ ]:
print("[Notebook 5/8] Validating environment and configuring inputs...")
import importlib.util
from pathlib import Path

_tensorflow_spec = importlib.util.find_spec("tensorflow")
if _tensorflow_spec is not None:
    raise RuntimeError(
        "TensorFlow is still importable after cleanup. Restart the Kaggle session after section 1; "
        f"origin={_tensorflow_spec.origin!r}, locations={_tensorflow_spec.submodule_search_locations!r}"
    )

MODEL_CONFIG = {
    "provider": "huggingface",
    "initialization": "config",
    "model_type": "llama",
    "dtype": "bfloat16",
    "config_overrides": {
        "vocab_size": 32064,
        "hidden_size": 1024,
        "intermediate_size": 2816,
        "num_hidden_layers": 8,
        "num_attention_heads": 8,
        "num_key_value_heads": 8,
        "max_position_embeddings": 2048,
        "tie_word_embeddings": True,
        "use_cache": False,
        "_attn_implementation": "sdpa",
    },
}
DATASET_ID = "LaughTaleAI/LaughLM-Tokenized-Fine"
DATASET_REVISION = "main"
TRAIN_SHARD_STOP = 1
DATA_CACHE_DIR = Path("/kaggle/working/huggingface-cache")
OUTPUT_DIR = Path("/kaggle/working/trainlm-llama-135m-mb8-ga8-chunked")
print("[Notebook 5/8] Inputs configured.")


## 3. Build the datasets and trainer

`from_hub()` downloads the requested `.bin` range once into `DATA_CACHE_DIR`, scans and validates it before TPU launch, then trains from those local files through lazy memory maps. It does not download during training. Users do not write download loops or configure a dataloader, process count, rank, or world size.


In [ ]:
print("[Notebook 6/8] Downloading/validating data and building trainer...")
import json
from trainlm import PackedBinDataset, TrainLMTrainer

sequence_length = 2048
train_dataset = PackedBinDataset.from_hub(
    DATASET_ID,
    revision=DATASET_REVISION,
    shard_range=(0, TRAIN_SHARD_STOP),
    sequence_length=sequence_length,
    cache_dir=DATA_CACHE_DIR,
    split="train",
)

trainer = TrainLMTrainer.from_config(
    {
        "api_version": "1",
        "model": MODEL_CONFIG,
        "training_args": {
            "output_dir": OUTPUT_DIR,
            "accelerator": "tpu",
            "bf16": True,
            "max_steps": 100,
            "sequence_length": sequence_length,
            "per_device_train_batch_size": 8,
            "gradient_accumulation_steps": 8,
            "logging_steps": 10,
            "warmup_steps": 5,
            "lr_scheduler_type": "wsd",
            "logging_verbosity": "verbose",
            "loss_implementation": "chunked_linear",
            "logits_chunk_size": 4096,
        },
    },
    train_dataset=train_dataset,
)
print("[Notebook 6/8] Trainer ready. No dataset download occurs during train().")


## 4. Train

This single public call launches one MB8/GA8 chunked-loss throughput experiment. It preserves the baseline's 1,048,576 scheduled tokens per update while halving the number of rank-local accumulation microsteps again. Evaluation and checkpointing are intentionally disabled so their separate compilation and I/O do not contaminate the training window.

This geometry is a candidate, not a promised 500K result. Full logits are still active, so stop and retain the established MB2/GA32 baseline if the target reports HBM exhaustion. A successful run must be compared using `summary.json` and `xla_metrics.txt`; do not infer performance from heartbeat timing. TrainLM owns the private worker launch and terminates its private worker process group on failure or interruption.


In [ ]:
print("[Notebook 7/8] Verifying the MB8/GA8 chunked-loss throughput candidate before TPU launch...")
_expected = {
    "output_dir": str(OUTPUT_DIR),
    "sequence_length": 2048,
    "per_device_train_batch_size": 8,
    "gradient_accumulation_steps": 8,
    "max_steps": 100,
    "warmup_steps": 5,
    "lr_scheduler_type": "wsd",
    "loss_implementation": "chunked_linear",
    "logits_chunk_size": 4096,
}
_actual = {
    "output_dir": str(trainer.args.output_dir),
    "sequence_length": trainer.args.sequence_length,
    "per_device_train_batch_size": trainer.args.per_device_train_batch_size,
    "gradient_accumulation_steps": trainer.args.gradient_accumulation_steps,
    "max_steps": trainer.args.max_steps,
    "warmup_steps": trainer.args.warmup_steps,
    "lr_scheduler_type": trainer.args.lr_scheduler_type,
    "loss_implementation": trainer.args.loss_implementation,
    "logits_chunk_size": trainer.args.logits_chunk_size,
}
if _actual != _expected:
    raise RuntimeError(
        "Stale notebook/trainer configuration detected. Restart the session, "
        f"reopen this notebook from the refreshed checkout, and rebuild the trainer. "
        f"Expected {_expected!r}; received {_actual!r}."
    )
if trainer.eval_dataset is not None or trainer.args.eval_steps is not None:
    raise RuntimeError(
        "The throughput candidate must not include evaluation. Rebuild the trainer "
        "from the current notebook before continuing."
    )
if trainer.args.save_steps is not None:
    raise RuntimeError(
        "The throughput candidate must not include checkpoint cadence. Rebuild the "
        "trainer from the current notebook before continuing."
    )
print("[Notebook 7/8] Throughput candidate verified:", _actual, flush=True)
print("[Notebook 7/8] Starting one private TPU lifecycle with a single live progress.md document (verbose progress updates in this cell).")
result = trainer.train()
print("[Notebook 7/8] TPU lifecycle completed.", flush=True)
_worker = result.get("worker_summary", {}) if isinstance(result, dict) else {}
_summary = {
    key: _worker.get(key)
    for key in (
        "phase", "steps", "micro_steps",
        "steady_global_supervised_tokens_per_second",
        "steady_global_scheduled_tokens_per_second",
        "measured_seconds_slowest_rank", "performance_certified",
    )
}
print("[Notebook 7/8] TPU summary:", json.dumps(_summary, sort_keys=True), flush=True)
del result, _worker, _summary


## 5. Optional: inspect what TrainLM selected

`explain()` is useful when reviewing fallbacks or filing a result. It does not require users to inspect worker commands or logs.

In [ ]:
print("[Notebook 8/8] Rendering optimization explanation...")
trainer.explain(format="text")

## 6. Optional lifecycle validation

After the 100-update throughput candidate succeeds, enable familiar `save_steps` and `eval_steps` arguments in a separate run. Keeping those operations out of this run makes the measured training window comparable.


In [ ]:
print(
    "[Optional lifecycle] The throughput candidate is complete. Add save_steps/eval_steps "
    "in a separate trainer configuration when validating those lifecycles."
)


## What to save from a validation run

Archive the output directory, including `coordinator_summary.json`, `summary.json`, `metrics.jsonl`, committed checkpoint manifests/shards, and XLA metrics. The returned result is the normal user-facing status; these files are only needed for debugging or performance certification.

A successful run validates the lifecycle on that TPU. It does not by itself mark performance as certified.